In [ ]:
##################################### IMPORTAMOS LIBRERIAS Y ENTORNO MODELO #############################################

from docplex.mp.model import Model
mdl=Model('modelo')
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd

In [ ]:
filename= r'C:\Users\renat\OneDrive\Escritorio\Practica\Tarjeta VRP\Instancias\A-n32-k5'#nombre del archivo mediante ubicación en  el PC

file = open(filename+'.txt') # abrir el archivo con su respectivo extensión, en este caso .txt
Data = file.readlines() # almacenar la infomracion del archivo en la memoria del computador, y lo almaceno con el nombre Data
file.close()

In [ ]:
#SACO LAS COORDENADAS DE LA INSTANCIA 
coord_x=[int(Data[i].split()[1]) for i in range(7,39)]
print('Coordenadas eje X=')
print(coord_x)
coord_y=[int(Data[i].split()[2]) for i in range(7,39)]
print('Coordenadas eje Y=')
print(coord_y)
q={i-40:int(Data[i].split()[1]) for i in range(40,72)}
print('Demanda de cliente')
print(q)
print('Pares Ordenados')
#Pares_coordenados={(int(Data[i].split()[1]),int(Data[i].split()[2])) for i in range(7,38)}
#print(Pares_coordenados)
Nodos=int(Data[3].split()[2])
print('Numero de Nodos')
print(Nodos)
Q=int(Data[5].split()[2])
print('Capacidad Q de vehiculos')
print(Q)

Clientes=[i for i in range(Nodos)] 
#Arcos entre ciudades
arcos =[(i,j) for i in Clientes for j in Clientes if i!=j]
distancia={(i, j): np.hypot(coord_x[i] - coord_x[j], coord_y[i] - coord_y[j]) for i,j in arcos}

In [ ]:
def modelo2_4(distancia, I, Q, q, k):

    ################################################## Rangos ##############################################################
    #(i,j)
    Secuencia1={(i,j) for i in range(I) for j in range(I)}
    Secuencia2={(i,j,k) for i in range(I) for j in range(I) for k in range(I)}
    Secuencia3={(i,j,k) for i in range(I) for j in range(I) for k in range(I)}

    ################################################### Variables ###########################################################
    #x_ij Si se va del nodo i al j.
    x=mdl.binary_var_dict(Secuencia1, name='x')
    #F_ijk 1 si se va del nodo i al j desde el deposito hasta k.
    F=mdl.binary_var_dict(Secuencia2, name='F')
    #G_ijk 1 si se va del nodo i al j desde k hasta el deposito.
    G=mdl.binary_var_dict(Secuencia3, name='G')

    ##################################################### FUNCIÓN OBJETIVO ###################################################

    #Minimiza los costos de oportunidad de dictar una asignatura i en un dia d.

    mdl.minimize(mdl.sum(distancia[i,j]*x[i,j] for i in range(I) for j in range(I) if i!=j))

    ########################################################### 2 ############################################################
    for i in range(I):
        if i!=0:
            mdl.add_constraint(mdl.sum(x[i,j] for j in range(I) if i!=j)==1)
        #print(mdl.add_constraint(mdl.sum(x[i,j] for j in range(I) if i!=j)==1))

    ########################################################### 3 ############################################################
    for i in range(I):
        if i!=0:
            mdl.add_constraint(mdl.sum(x[j,i] for j in range(I) if i!=j)==1)
    
    ########################################################### EXTRA ############################################################
    mdl.add_constraint(mdl.sum(x[0,j] for j in range(I) if i!=j)==k)

    ########################################################### EXTRA II ############################################################
    mdl.add_constraint(mdl.sum(x[j,0] for j in range(I) if i!=j)==k)


    ########################################################### 25 ############################################################
    for k in range(I):
        if k!=0:
            mdl.add_constraint(mdl.sum(F[0,j,k] for j in range(I) if j!=0)-mdl.sum(F[j,0,k] for j in range(I) if j!=0)==1)

    ########################################################### 26 ############################################################
    for k in range(I):
        for i in range(I):
            if i!=k:
                if k!=0:
                    if i!=0:
                        mdl.add_constraint(mdl.sum(F[i,j,k] for j in range(I) if i!=j)-mdl.sum(F[j,i,k] for j in range(I) if j!=i)==0)

    ########################################################### 27 ############################################################
    for k in range(I):
        if k!=0:
            mdl.add_constraint(mdl.sum(G[j,0,k] for j in range(I) if j!=0)-mdl.sum(G[0,j,k] for j in range(I) if j!=0)==1)

    ########################################################### 28 ############################################################
    for k in range(I):
        for i in range(I):
            if i!=k:
                if k!=0:
                    if i!=0:
                        mdl.add_constraint(mdl.sum(G[i,j,k] for j in range(I) if i!=j)-mdl.sum(G[j,i,k] for j in range(I) if i!=j)==0)

    ########################################################### 29 ############################################################
    for i in range(I):
        for j in range(I):
            for k in range(I):
                if i!=j:
                    if k!=0:
                        mdl.add_constraint(F[i,j,k]+G[i,j,k]<=x[i,j])

    ########################################################### 30 ############################################################
    for i in range(I):
        for j in range(I):
            if i!=j:
                mdl.add_constraint(mdl.sum(q[k]*(F[i,j,k]+G[i,j,k]) for k in range(I) if k!=0)<=(Q-q[i]-q[j])*x[i,j])

    for i in range(I):
        mdl.add_constraint(x[i,i]==0)
    
    for i in range(I):
        for j in range(I):
            mdl.add_constraint(F[i,i,j]==0)
    
    for i in range(I):
        for j in range(I):
            mdl.add_constraint(G[i,i,j]==0)

    mdl.parameters.timelimit=60
    solucion=mdl.solve(log_output=True)
    solucion.display()

    plt.figure(figsize=(20,15))#Tamaño de la figura
    plt.xlabel("Distancia X")#Nombre del eje x
    plt.ylabel("Distancia Y")#Nombre del eje y
    plt.title("Ubicación de los clientes")#Titulo 

    #Para cuando esta la solucion
    arcos_activos = [i for i in arcos if x[i].solution_value > 0.9]
    #para los arcos activos, las lineas seran de color azul
    for i,j in arcos_activos:
        plt.plot([coord_x[i],coord_x[j]],[coord_y[i],coord_y[j]],
                  color='b', alpha=0.4, zorder=0)
    plt.scatter(x=coord_x, y=coord_y, color='blue', zorder=1)
    #Mostrara el numero del nodo     
    for n in range(len(coord_x)):
        plt.annotate(str(n), xy=(coord_x[n],coord_y[n] ), 
                     xytext=(coord_x[n]+0.5,coord_y[n]+1),color='red')

    plt.xlim((0,100))#de donde a donde va el eje x
    plt.ylim((0,100))#de donde a donde va el eje y
    plt.show()# Muestra el grafico 

In [ ]:
modelo2_4(distancia, 32, 100, q, 5 )